<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Solution: Keep the index fresh

**[Keep the index fresh](https://learning.nextia-ai.com/courses/search/m08/keep-the-index-fresh/)** · Search and Ranking: Find the Best Match · Module 8, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find what a stale index misses, rebuild it, update price and stock without a rebuild, hide removed products, and switch to a new index version only after it passes a check.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (about 11 MB). No account, no key and no model download. |
| **You should know** | How BM25 ranks products, from [BM25](https://learning.nextia-ai.com/courses/search/m03/bm25/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m08/keep-the-index-fresh/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M08-L03-keep-the-index-fresh/keep-the-index-fresh-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, bm25s 0.3.16, PyStemmer 3.1.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2 bm25s==0.3.16 PyStemmer==3.1.0
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'bm25s', 'PyStemmer']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 10 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/catalogue.csv.gz": "569c1c4c41403d59492e4cb22be61a7e634b052f67b4d32d12a9f14e77c08c52",
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Yesterday's index

Larkfield adds new products every day. Our index was built yesterday, without this week's products. Run the cell: it builds that index and counts what it misses. The "added" dates are **synthetic**: our code made them.

In [ ]:
import re, time, bm25s
cat = catalogue()
q = queries()
j = judgments()
tok = lambda t: [w for w in re.split(r"[^\w]+", str(t).lower()) if w]

def build_index(products):
    """A BM25 index of title and description, and the product IDs in index order."""
    r = bm25s.BM25(k1=1.2, b=0.75, method="lucene")
    r.index([tok(t + " " + d) for t, d in zip(products["title"], products["description"])], show_progress=False)
    return r, products["product_id"].to_numpy()

def search(index, text, k=10, hide=()):
    r, ids = index
    words = [w for w in tok(text) if w in r.vocab_dict]
    if not words:
        return []
    s = r.get_scores(words)
    top = [ids[i] for i in np.argsort(-s, kind="stable")[: k + len(hide)] if s[i] > 0]
    return [p for p in top if p not in hide][:k]

new = cat["added"] >= "2025-10-20"          # added in the last week of the log (synthetic dates)
t0 = time.perf_counter()
old_index = build_index(cat[~new])
print(f"Yesterday's index: {len(old_index[1]):,} products, built in {time.perf_counter() - t0:.1f} s")
print(f"Not in it: {new.sum()} products added this week")

> **Check.** Yesterday's index has 39791 products. 275 products added this week are not in it. A product that is not in the index cannot be found, however good it is.

## 2. Who misses the new products?

In [ ]:
new_ids = set(cat.loc[new, "product_id"])
eng = q[(q["split"].isin(["validation", "test"])) & (q["language"] == "en")]
cases = [(qid, text) for qid, text in zip(eng["query_id"], eng["query"])
         if any(g == 2 and p in new_ids for p, g in j.get(qid, {}).items())]
print(len(cases), "English validation and test queries have an exact product from this week")
qid, text = cases[0]
print(text, "->", [p for p in search(old_index, text) if p in new_ids], "(new products in yesterday's top 10)")

> **Check.** 54 English validation and test queries have an exact product from this week. For "home sweet home sign", yesterday's top 10 has no new product at all.

## 3. Your turn: rebuild the index

Build today's index from the whole catalogue, and count the queries that now show an exact new product in their top 10.

In [ ]:
t0 = time.perf_counter()
new_index = build_index(cat)
print(f"Today's index: {len(new_index[1]):,} products, built in {time.perf_counter() - t0:.1f} s")
found = sum(any(p in new_ids and j[qid].get(p) == 2 for p in search(new_index, text)) for qid, text in cases)
print(found, "of", len(cases), "queries now show an exact new product in the top 10")

In [ ]:
expect('found', found, 7)

> **Check.** 7 of 54 queries. Look at the build time: a few seconds for 40,066 products. At this size, a full rebuild every night (or every hour) is the simplest correct answer.

## 4. Fields that change every minute

Price, stock and the sale flag change far more often than titles. Do not rebuild the text index for them. Keep them in a small table and read it when you build the page.

In [ ]:
# Price and stock change often. Keep them out of the text index, in a small table you update in place.
stock = dict(zip(cat["product_id"], cat["in_stock"]))
first = search(new_index, "outdoor cushion", k=3)
print([(p, stock[p]) for p in first])
stock[first[0]] = False                   # the warehouse sold the last one: no rebuild needed
print("in stock now:", [p for p in first if stock[p]])

## 5. Removed products

A removed product must leave the page at once, not tomorrow. Until the next rebuild, hide it at query time. Such a "hidden until removed" mark is called a **tombstone**.

In [ ]:
# A removed product must disappear at once. Hide it at query time until the next rebuild removes it.
removed = {first[0]}
print(search(new_index, "outdoor cushion", k=3, hide=removed))

## 6. Versions and a pointer

Never build the new index over the one that serves people. Build each version in its own folder. A small file, `current`, names the version in use. The search reads `current` before it opens an index.

In [ ]:
import os, shutil, json
from pathlib import Path
root = Path("index-versions"); shutil.rmtree(root, ignore_errors=True); root.mkdir()

def save_version(index, name):
    r, ids = index
    r.save(str(root / name)); np.save(root / name / "ids.npy", ids)
    (root / name / "manifest.json").write_text(json.dumps({"products": len(ids), "k1": 1.2, "b": 0.75}))

def current():
    return (root / "current").read_text()

save_version(old_index, "v41"); (root / "current").write_text("v41")
save_version(new_index, "v42")
print("serving:", current(), "| folders:", sorted(p.name for p in root.iterdir()))

## 7. Your turn: switch only after a check

Complete `switch`: refuse a version whose product count is not the one you expect. The switch itself is one atomic step: `os.replace` changes the pointer file in one move, so no search ever reads half a file.

In [ ]:
def switch(name, expected_products):
    """Point `current` to a new version, only if its checks pass. One atomic step: os.replace."""
    m = json.loads((root / name / "manifest.json").read_text())
    if m["products"] != expected_products:
        return False
    tmp = root / "current.tmp"
    tmp.write_text(name)
    os.replace(tmp, root / "current")
    return True

print(switch("v42", expected_products=len(cat) - 1), current())    # a wrong count: refused
print(switch("v42", expected_products=len(cat)), current())        # the right count: switched
serving = current()

In [ ]:
expect('serving', serving, 'v42')

> **Check.** The wrong count was refused, and the right one switched `current` to v42. Version v41 is still on disk: to roll back, point `current` to v41 again. In the project, `LARKFIELD_OUTPUTS=outputs-next search index` builds a new version in its own folder in the same way.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [The same ideas in a search server](https://learning.nextia-ai.com/courses/search/m08/the-same-ideas-in-a-search-server/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m08/keep-the-index-fresh/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.